# Student Performance Prediction
## Notebook 2 — Exploratory Data Analysis (EDA)

**Author:** Abdifatah Muhlar
**Data Source:** UCI Machine Learning Repository — Student Performance Dataset

---

### Objective
This notebook performs exploratory data analysis on the student
performance dataset. We examine grade distributions, feature
correlations, and the relationship between behavioral/demographic
factors and academic outcomes — extracting educational insights
that inform both our machine learning model and real-world
policy recommendations.

In [5]:
# ============================================================
# SECTION 1 — Import Libraries & Load Data
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load clean dataset
df = pd.read_csv('student_clean.csv')

print("Dataset loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())

Dataset loaded successfully
Shape: (395, 34)

First 5 rows:
   school  sex  age  address  famsize  Pstatus  Medu  Fedu  Mjob  Fjob  ...  \
0       0    0   18        1        0        0     4     4     0     4  ...   
1       0    0   17        1        0        1     1     1     0     2  ...   
2       0    0   15        1        1        1     1     1     0     2  ...   
3       0    0   15        1        0        1     4     2     1     3  ...   
4       0    0   16        1        0        1     3     3     2     2  ...   

   freetime  goout  Dalc  Walc  health  absences  G1  G2  G3  Pass  
0         3      4     1     1       3         6   5   6   6     0  
1         3      3     1     1       3         4   5   5   6     0  
2         3      2     2     3       3        10   7   8  10     1  
3         2      2     1     1       5         2  15  14  15     1  
4         3      2     1     2       5         4   6  10  10     1  

[5 rows x 34 columns]


---
## Section 2 — Grade Distribution Analysis

We examine the distribution of final grades (G3) and the
pass/fail breakdown to understand the overall academic
performance landscape in the dataset.

In [6]:
# ============================================================
# SECTION 2 — Grade Distribution Analysis
# ============================================================

from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Final Grade (G3) Distribution', 'Pass/Fail Breakdown'),
    specs=[[{'type': 'xy'}, {'type': 'domain'}]]
)

# Grade distribution
fig.add_trace(go.Histogram(
    x=df['G3'],
    nbinsx=20,
    marker_color='royalblue',
    name='Grade Distribution',
    hovertemplate='Grade: %{x}<br>Count: %{y}<extra></extra>'
), row=1, col=1)

# Pass/Fail pie
fig.add_trace(go.Pie(
    labels=['Pass', 'Fail'],
    values=df['Pass'].value_counts().values,
    marker_colors=['steelblue', 'crimson'],
    hovertemplate='%{label}: %{value} students (%{percent})<extra></extra>'
), row=1, col=2)

fig.update_layout(
    title=dict(
        text='Student Final Grade Distribution — Mathematics',
        font=dict(size=16), x=0.5, xanchor='center'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False
)

fig.update_xaxes(title_text='Final Grade (0-20)', row=1, col=1)
fig.update_yaxes(title_text='Number of Students', row=1, col=1)

fig.show()
print("Chart rendered")

Chart rendered


### Insight — Grade Distribution

Observed Pattern:
- Grade distribution is roughly bimodal — one cluster around
  8-10 (borderline students) and another around 12-15 (solid performers)
- No student scored below 3, suggesting a grading floor effect
- 67.1% of students passed — class balance is acceptable for modeling

Educational Interpretation:
The bimodal distribution is significant. It suggests two distinct
student populations exist within the same classroom:

1. Struggling students — clustered around the pass/fail boundary (8-10)
   These students are most at risk and represent the primary target
   for early intervention. A predictive model identifying these
   students early could allow teachers to provide targeted support
   before final exams.

2. Performing students — clustered around 12-15
   These students demonstrate consistent academic capability.
   Understanding what differentiates them from struggling peers
   is the core question this project addresses.

Implication:
A one-size-fits-all teaching approach is unlikely to serve both
groups effectively. Data-driven identification of at-risk students
enables personalized intervention — the practical application of
this prediction model.

---
## Section 3 — Key Feature Analysis

We examine the relationship between the most important behavioral
and demographic features and student pass/fail outcomes.

In [7]:
# ============================================================
# SECTION 3 — Key Feature Analysis
# ============================================================

fig = make_subplots(rows=2, cols=2,
    subplot_titles=(
        'Study Time vs Pass/Fail',
        'Past Failures vs Pass/Fail',
        'Mother Education vs Pass/Fail',
        'Absences vs Pass/Fail'
    ))

# Color map
colors = {0: 'crimson', 1: 'steelblue'}
labels = {0: 'Fail', 1: 'Pass'}

for pass_val in [0, 1]:
    subset = df[df['Pass'] == pass_val]

    # Study time
    fig.add_trace(go.Box(
        y=subset['studytime'],
        name=labels[pass_val],
        marker_color=colors[pass_val],
        showlegend=True if pass_val in [0,1] else False,
        hovertemplate=f'{labels[pass_val]}<br>Study Time: %{{y}}<extra></extra>'
    ), row=1, col=1)

    # Past failures
    fig.add_trace(go.Box(
        y=subset['failures'],
        name=labels[pass_val],
        marker_color=colors[pass_val],
        showlegend=False,
        hovertemplate=f'{labels[pass_val]}<br>Failures: %{{y}}<extra></extra>'
    ), row=1, col=2)

    # Mother education
    fig.add_trace(go.Box(
        y=subset['Medu'],
        name=labels[pass_val],
        marker_color=colors[pass_val],
        showlegend=False,
        hovertemplate=f'{labels[pass_val]}<br>Medu: %{{y}}<extra></extra>'
    ), row=2, col=1)

    # Absences
    fig.add_trace(go.Box(
        y=subset['absences'],
        name=labels[pass_val],
        marker_color=colors[pass_val],
        showlegend=False,
        hovertemplate=f'{labels[pass_val]}<br>Absences: %{{y}}<extra></extra>'
    ), row=2, col=2)

fig.update_layout(
    title=dict(
        text='Key Features vs Pass/Fail Outcome',
        font=dict(size=16), x=0.5, xanchor='center'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    boxmode='group',
    height=600
)

fig.show()
print("Chart rendered")

Chart rendered


### Insight — Key Feature Analysis

Observed Patterns:
- Study time: Passing students show higher median study time
- Past failures: Failing students have significantly more past failures
- Mother education: Higher maternal education correlates with passing
- Absences: Failing students tend to have more absences

Educational Interpretation:
Each feature tells a distinct story about student performance:

1. Study Time
   The positive relationship between study time and passing is
   expected but important to quantify. However the overlap between
   pass and fail groups suggests study time alone is insufficient —
   quality of study and support systems matter equally. Students
   studying less than 2 hours per week are at significantly higher
   risk of failing.

2. Past Failures
   This is the strongest single predictor visible in the data.
   Students with prior failures carry compounding disadvantages —
   lost confidence, knowledge gaps, and reduced motivation. This
   finding supports early intervention programs that prevent first
   failures rather than remediate after them.

3. Mother's Education Level
   Parental education level reflects the home learning environment.
   Students with highly educated mothers benefit from homework support,
   educational encouragement, and exposure to academic culture at home.
   This highlights an equity dimension — students from less educated
   households face structural disadvantages that schools must actively
   compensate for.

4. Absences
   Higher absences strongly correlate with failure. Each missed class
   represents lost instruction time and signals disengagement.
   Attendance monitoring combined with early outreach to at-risk
   students could prevent a significant proportion of failures.

Implication:
These four features — study habits, academic history, family background,
and attendance — form the core predictive profile of a struggling student.
A school intervention system built on these signals could identify
at-risk students weeks before final exams, enabling targeted support.

---
## Section 4 — Correlation Heatmap

We examine correlations between all numeric features to identify
which variables are most strongly related to the Pass/Fail outcome
and to detect any multicollinearity between features.

In [8]:
# ============================================================
# SECTION 4 — Correlation Heatmap
# ============================================================

import plotly.figure_factory as ff
import numpy as np

# Select numeric columns including Pass
numeric_cols = ['studytime', 'failures', 'absences', 'Medu', 'Fedu',
                'age', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc',
                'health', 'traveltime', 'Pass']

corr = df[numeric_cols].corr().round(2)

# Lower triangle only
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
corr_masked = corr.copy()
corr_masked[mask] = np.nan

z = corr_masked.values.tolist()
labels = list(corr.columns)

custom_colorscale = [
    [0.0,  '#2166ac'],
    [0.25, '#92c5de'],
    [0.5,  '#f7f7f7'],
    [0.75, '#f4a582'],
    [1.0,  '#b2182b'],
]

fig = go.Figure(data=go.Heatmap(
    z=z,
    x=labels,
    y=labels,
    colorscale=custom_colorscale,
    zmin=-1,
    zmax=1,
    colorbar=dict(
        title=dict(text='Correlation', font=dict(size=12)),
        tickvals=[-1, -0.5, 0, 0.5, 1],
        ticktext=['-1.0', '-0.5', '0.0', '0.5', '1.0']
    ),
    hovertemplate='%{y} vs %{x}<br>Correlation: %{z:.2f}<extra></extra>'
))

for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr_masked.iloc[i, j]
        if not np.isnan(val):
            text_color = 'white' if abs(val) > 0.6 else 'black'
            fig.add_annotation(
                x=labels[j], y=labels[i],
                text=f'{val:.2f}',
                showarrow=False,
                font=dict(size=10, color=text_color)
            )

fig.update_layout(
    title=dict(
        text='Correlation Heatmap — Student Performance Features',
        font=dict(size=16), x=0.5, xanchor='center'
    ),
    xaxis=dict(tickfont=dict(size=10), side='bottom',
               showgrid=False, tickangle=45),
    yaxis=dict(tickfont=dict(size=10), autorange='reversed', showgrid=False),
    width=750,
    height=650,
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()
print("Heatmap rendered")

Heatmap rendered


### Insight — Correlation Heatmap

Key Correlations with Pass/Fail Outcome:
- failures vs Pass : -0.34 — strongest negative predictor
- studytime vs Pass: positive — more study time linked to passing
- absences vs Pass : negative — more absences linked to failing
- Medu vs Pass     : positive — higher maternal education helps

Notable Feature Relationships:
- Medu vs Fedu = 0.62 — parents tend to have similar education levels
- Dalc vs Walc = 0.65 — students who drink on weekdays also drink on weekends
- goout vs Walc = 0.42 — social outings correlate with weekend drinking

Educational Interpretation:
The correlation matrix reveals three distinct factor groups that
influence student performance:

1. Academic History Factors
   Past failures is the single strongest correlate with the Pass/Fail
   outcome. This confirms what educational research consistently shows —
   prior academic failure is the most reliable early warning signal.
   Schools that track failure history and intervene immediately after
   first failures prevent the compounding effect seen in repeat failures.

2. Family Background Factors
   Parental education levels (Medu, Fedu) are positively correlated
   with passing. The strong correlation between Medu and Fedu (0.62)
   suggests these variables carry overlapping information — both
   reflecting the overall home learning environment. Students from
   households where neither parent completed higher education face
   a measurable structural disadvantage.

3. Social Behavior Factors
   The Dalc-Walc-goout cluster reveals a social risk pattern. Students
   with high alcohol consumption and frequent social outings show weaker
   academic performance. This is not simply about alcohol — it reflects
   a broader pattern of time allocation. Students investing heavily in
   social activities have less time and cognitive capacity for academic work.

Multicollinearity Note:
The Medu-Fedu correlation (0.62) and Dalc-Walc correlation (0.65)
indicate some multicollinearity. This is acceptable for tree-based
models like Random Forest but worth noting for Logistic Regression
interpretation.

---
## Section 5 — Social & Behavioral Risk Analysis

We examine how social behavior patterns — alcohol consumption,
going out, and romantic relationships — relate to academic outcomes.
These factors are often overlooked in traditional academic assessments
but carry meaningful predictive signal.

In [10]:
# ============================================================
# SECTION 5 — Social & Behavioral Risk Analysis
# ============================================================

fig = make_subplots(rows=1, cols=3,
    subplot_titles=(
        'Alcohol Consumption',
        'Going Out Frequency',
        'Romantic Relationship'
    ),
    horizontal_spacing=0.12)

# Alcohol consumption (average of Dalc and Walc)
df['avg_alcohol'] = (df['Dalc'] + df['Walc']) / 2
alcohol_pass = df[df['Pass']==1]['avg_alcohol'].mean().round(2)
alcohol_fail = df[df['Pass']==0]['avg_alcohol'].mean().round(2)

fig.add_trace(go.Bar(
    x=['Pass', 'Fail'],
    y=[alcohol_pass, alcohol_fail],
    marker_color=['steelblue', 'crimson'],
    hovertemplate='%{x}<br>Avg Alcohol Score: %{y:.2f}<extra></extra>',
    name='Alcohol'
), row=1, col=1)

# Going out
goout_pass = df[df['Pass']==1]['goout'].mean().round(2)
goout_fail = df[df['Pass']==0]['goout'].mean().round(2)

fig.add_trace(go.Bar(
    x=['Pass', 'Fail'],
    y=[goout_pass, goout_fail],
    marker_color=['steelblue', 'crimson'],
    hovertemplate='%{x}<br>Avg Goout Score: %{y:.2f}<extra></extra>',
    name='Goout'
), row=1, col=2)

# Romantic relationship
romantic_pass = df[df['Pass']==1]['romantic'].mean().round(2)
romantic_fail = df[df['Pass']==0]['romantic'].mean().round(2)

fig.add_trace(go.Bar(
    x=['Pass', 'Fail'],
    y=[romantic_pass, romantic_fail],
    marker_color=['steelblue', 'crimson'],
    hovertemplate='%{x}<br>Proportion: %{y:.2f}<extra></extra>',
    name='Romantic'
), row=1, col=3)

fig.update_layout(
    title=dict(
        text='Social & Behavioral Factors vs Academic Outcome',
        font=dict(size=16), x=0.5, xanchor='center'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    height=500,
    width=900,
    margin=dict(t=100)
)

fig.update_yaxes(showgrid=True, gridcolor='lightgrey')
fig.update_annotations(font_size=13)

fig.show()
print("Chart rendered")

Chart rendered


In [11]:
print("VERIFICATION — Social & Behavioral Factors")
print("=" * 45)
print(f"\nAlcohol Consumption (avg of Dalc + Walc):")
print(f"  Pass students : {alcohol_pass}")
print(f"  Fail students : {alcohol_fail}")

print(f"\nGoing Out Frequency:")
print(f"  Pass students : {goout_pass}")
print(f"  Fail students : {goout_fail}")

print(f"\nRomantic Relationship (proportion):")
print(f"  Pass students : {romantic_pass}")
print(f"  Fail students : {romantic_fail}")

VERIFICATION — Social & Behavioral Factors

Alcohol Consumption (avg of Dalc + Walc):
  Pass students : 1.85
  Fail students : 1.95

Going Out Frequency:
  Pass students : 2.97
  Fail students : 3.4

Romantic Relationship (proportion):
  Pass students : 0.3
  Fail students : 0.4


### Insight — Social & Behavioral Risk Analysis

Verified Statistics:
- Alcohol: Pass = 1.85, Fail = 1.95 — small but consistent difference
- Going Out: Pass = 2.97, Fail = 3.40 — strongest behavioral signal
- Romantic: Pass = 0.30, Fail = 0.40 — modest but present difference

Educational Interpretation:
These three behavioral factors reveal important patterns in student
risk profiles that go beyond academic ability:

1. Alcohol Consumption (difference = 0.10)
   The difference is small but consistent. Alcohol consumption alone
   is not a strong predictor — however it forms part of a broader
   behavioral risk cluster. Students showing high alcohol consumption
   alongside frequent social outings represent a compounded risk profile
   that is more predictive than any single factor alone.

2. Going Out Frequency (difference = 0.43)
   This is the strongest behavioral signal in the dataset. Failing
   students go out significantly more frequently than passing students.
   This reflects a time allocation pattern — hours spent on social
   activities outside school directly compete with study time and rest.
   The effect is compounded when outings extend into weekday evenings,
   disrupting sleep and reducing classroom engagement.

3. Romantic Relationships (difference = 0.10)
   A modest but present difference. Romantic relationships during
   adolescence can introduce emotional stress and time demands that
   distract from academic focus — particularly among students who are
   already academically vulnerable.

Implication:
Social and behavioral factors are actionable intervention points.
Unlike family background or parental education which schools cannot
change, behavioral patterns can be addressed through counseling,
structured after-school programs, and student engagement initiatives.
Going out frequency is the most actionable signal — schools could
use this alongside attendance data to build an early warning system
for at-risk students.

In [12]:
from google.colab import files
files.download('student_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
!pip install kaleido==0.2.1 --quiet

In [18]:
!pip install kaleido==0.2.1

In [19]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('student_clean.csv')
df['avg_alcohol'] = (df['Dalc'] + df['Walc']) / 2

# --- CHART 1: Grade Distribution ---
fig1 = make_subplots(rows=1, cols=2,
    subplot_titles=('Final Grade (G3) Distribution', 'Pass/Fail Breakdown'),
    specs=[[{'type': 'xy'}, {'type': 'domain'}]])

fig1.add_trace(go.Histogram(
    x=df['G3'], nbinsx=20,
    marker_color='royalblue',
    hovertemplate='Grade: %{x}<br>Count: %{y}<extra></extra>'
), row=1, col=1)

fig1.add_trace(go.Pie(
    labels=['Pass', 'Fail'],
    values=df['Pass'].value_counts().values,
    marker_colors=['steelblue', 'crimson'],
    hovertemplate='%{label}: %{value} students (%{percent})<extra></extra>'
), row=1, col=2)

fig1.update_layout(
    title=dict(text='Student Final Grade Distribution — Mathematics',
               font=dict(size=16), x=0.5, xanchor='center'),
    plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig1.update_xaxes(title_text='Final Grade (0-20)', row=1, col=1)
fig1.update_yaxes(title_text='Number of Students', row=1, col=1)
fig1.write_html('eda_grade_distribution.html')
print("eda_grade_distribution.html saved")

# --- CHART 2: Key Features Box Plots ---
fig2 = make_subplots(rows=2, cols=2,
    subplot_titles=('Study Time vs Pass/Fail', 'Past Failures vs Pass/Fail',
                    'Mother Education vs Pass/Fail', 'Absences vs Pass/Fail'))

colors = {0: 'crimson', 1: 'steelblue'}
labels = {0: 'Fail', 1: 'Pass'}
features = [('studytime', 1, 1), ('failures', 1, 2),
            ('Medu', 2, 1), ('absences', 2, 2)]

for feat, row, col in features:
    for pass_val in [0, 1]:
        subset = df[df['Pass'] == pass_val]
        fig2.add_trace(go.Box(
            y=subset[feat],
            name=labels[pass_val],
            marker_color=colors[pass_val],
            showlegend=True if (feat == 'studytime') else False,
            hovertemplate=f'{labels[pass_val]}<br>{feat}: %{{y}}<extra></extra>'
        ), row=row, col=col)

fig2.update_layout(
    title=dict(text='Key Features vs Pass/Fail Outcome',
               font=dict(size=16), x=0.5, xanchor='center'),
    plot_bgcolor='white', paper_bgcolor='white',
    boxmode='group', height=600)
fig2.write_html('eda_key_features.html')
print("eda_key_features.html saved")

# --- CHART 3: Correlation Heatmap ---
numeric_cols = ['studytime', 'failures', 'absences', 'Medu', 'Fedu',
                'age', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc',
                'health', 'traveltime', 'Pass']
corr = df[numeric_cols].corr().round(2)
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
corr_masked = corr.copy()
corr_masked[mask] = np.nan
z = corr_masked.values.tolist()
labels_heatmap = list(corr.columns)

custom_colorscale = [
    [0.0, '#2166ac'], [0.25, '#92c5de'],
    [0.5, '#f7f7f7'], [0.75, '#f4a582'], [1.0, '#b2182b']]

fig3 = go.Figure(data=go.Heatmap(
    z=z, x=labels_heatmap, y=labels_heatmap,
    colorscale=custom_colorscale, zmin=-1, zmax=1,
    colorbar=dict(title=dict(text='Correlation', font=dict(size=12)),
                  tickvals=[-1,-0.5,0,0.5,1],
                  ticktext=['-1.0','-0.5','0.0','0.5','1.0']),
    hovertemplate='%{y} vs %{x}<br>Correlation: %{z:.2f}<extra></extra>'
))

for i in range(len(labels_heatmap)):
    for j in range(len(labels_heatmap)):
        val = corr_masked.iloc[i, j]
        if not np.isnan(val):
            text_color = 'white' if abs(val) > 0.6 else 'black'
            fig3.add_annotation(
                x=labels_heatmap[j], y=labels_heatmap[i],
                text=f'{val:.2f}', showarrow=False,
                font=dict(size=10, color=text_color))

fig3.update_layout(
    title=dict(text='Correlation Heatmap — Student Performance Features',
               font=dict(size=16), x=0.5, xanchor='center'),
    xaxis=dict(tickfont=dict(size=10), side='bottom',
               showgrid=False, tickangle=45),
    yaxis=dict(tickfont=dict(size=10), autorange='reversed', showgrid=False),
    width=750, height=650,
    plot_bgcolor='white', paper_bgcolor='white')
fig3.write_html('eda_heatmap.html')
print("eda_heatmap.html saved")

# --- CHART 4: Social Behavioral ---
alcohol_vals = [df[df['Pass']==1]['avg_alcohol'].mean().round(2),
                df[df['Pass']==0]['avg_alcohol'].mean().round(2)]
goout_vals = [df[df['Pass']==1]['goout'].mean().round(2),
              df[df['Pass']==0]['goout'].mean().round(2)]
romantic_vals = [df[df['Pass']==1]['romantic'].mean().round(2),
                 df[df['Pass']==0]['romantic'].mean().round(2)]

fig4 = make_subplots(rows=1, cols=3,
    subplot_titles=('Alcohol Consumption', 'Going Out Frequency',
                    'Romantic Relationship'),
    horizontal_spacing=0.12)

for col, vals in zip([1, 2, 3],
    [alcohol_vals, goout_vals, romantic_vals]):
    fig4.add_trace(go.Bar(
        x=['Pass', 'Fail'], y=vals,
        marker_color=['steelblue', 'crimson'],
        text=[f'{v:.2f}' for v in vals],
        textposition='outside'
    ), row=1, col=col)

fig4.update_layout(
    title=dict(text='Social & Behavioral Factors vs Academic Outcome',
               font=dict(size=16), x=0.5, xanchor='center'),
    plot_bgcolor='white', paper_bgcolor='white',
    showlegend=False, height=500, width=900, margin=dict(t=100))
fig4.update_yaxes(showgrid=True, gridcolor='lightgrey')
fig4.update_annotations(font_size=13)
fig4.write_html('eda_social_behavioral.html')
print("eda_social_behavioral.html saved")

print("\nAll 4 charts saved as HTML")

eda_grade_distribution.html saved
eda_key_features.html saved
eda_heatmap.html saved
eda_social_behavioral.html saved

All 4 charts saved as HTML


In [20]:
from google.colab import files
files.download('eda_grade_distribution.html')
files.download('eda_key_features.html')
files.download('eda_heatmap.html')
files.download('eda_social_behavioral.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
files.download('eda_grade_distribution.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>